# Capa Gold — KPIs para el dashboard

## 1. Setup — Lectura de tablas Silver

In [0]:
from pyspark.sql import functions as F

df_pacientes = spark.table("workspace.silver.pacientes")
df_citas = spark.table("workspace.silver.citas")
df_eventos = spark.table("workspace.silver.eventos_clinicos")
df_facturacion = spark.table("workspace.silver.facturacion")

spark.sql("CREATE SCHEMA IF NOT EXISTS workspace.gold")

## 2. KPI: Citas por especialidad (volumen + tasa de inasistencia)

In [0]:
# Por especialidad: total de citas, cuántas fueron "No asistió" o "Cancelada", y el % que eso representa
df_citas_por_especialidad = (
    df_citas
    .groupBy("especialidad")
    .agg(
        F.count("*").alias("total_citas"),
        F.sum(
            F.when(F.col("estado_cita").isin("No asistió", "Cancelada"), 1).otherwise(0)
        ).alias("citas_no_efectivas")
    )
    .withColumn(
        "tasa_inasistencia_pct",
        F.round(F.col("citas_no_efectivas") / F.col("total_citas") * 100, 2)
    )
    .orderBy(F.desc("total_citas"))
)

display(df_citas_por_especialidad)

## 3. KPI: Facturación mensual (total y promedio)

In [0]:
# Agrupamos por año-mes de la factura
df_facturacion_mensual = (
    df_facturacion
    .withColumn("anio_mes", F.date_format("fecha_factura", "yyyy-MM"))
    .groupBy("anio_mes")
    .agg(
        F.round(F.sum("valor_neto"), 2).alias("total_facturado"),
        F.round(F.avg("valor_neto"), 2).alias("promedio_factura"),
        F.count("*").alias("cantidad_facturas")
    )
    .orderBy("anio_mes")
)

display(df_facturacion_mensual)

## 4. KPI: Eventos clínicos por tipo

In [0]:
from pyspark.sql import Window

# Distribución de tipos de evento (signos vitales, procedimientos, diagnósticos, etc.)
df_eventos_por_tipo = (
    df_eventos
    .groupBy("tipo_evento")
    .agg(F.count("*").alias("total_eventos"))
    .withColumn(
        "porcentaje_del_total",
        F.round(F.col("total_eventos") / F.sum("total_eventos").over(Window.partitionBy()) * 100, 2)
    )
    .orderBy(F.desc("total_eventos"))
)

display(df_eventos_por_tipo)

## 5. Guardar tablas Gold

In [0]:
df_citas_por_especialidad.write.mode("overwrite").format("delta").saveAsTable("workspace.gold.citas_por_especialidad")
df_facturacion_mensual.write.mode("overwrite").format("delta").saveAsTable("workspace.gold.facturacion_mensual")
df_eventos_por_tipo.write.mode("overwrite").format("delta").saveAsTable("workspace.gold.eventos_por_tipo")

print("Capa Gold completa")

## 6. Verificación final

In [0]:
spark.sql("SHOW TABLES IN workspace.gold").show(truncate=False)

for tabla in ["citas_por_especialidad", "facturacion_mensual", "eventos_por_tipo"]:
    total = spark.table(f"workspace.gold.{tabla}").count()
    print(f"{tabla}: {total} filas")